In [1]:
import torch
from llmfromscratch.llm_arch import GPTModel

In [2]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,    #1
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,       #2
    "qkv_bias": False
}

In [3]:
torch.manual_seed(123)

In [5]:
model = GPTModel(GPT_CONFIG_124M)
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trns_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_feature

## 5.1 Evaluating Generative Text Models

In [11]:
import tiktoken
from llmfromscratch.llm_gen import generate_text_simplified

def text_to_token_ids(text, tokenizer):
    token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    encoded_tensor = torch.tensor(token_ids).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flatten_text = token_ids.squeeze(0).tolist()
    decoded_text = tokenizer.decode(flatten_text)
    return decoded_text

In [12]:
start_text = "Every effort moves you closer to"
tokenizer = tiktoken.get_encoding("gpt2")
encoded_tensor = text_to_token_ids(start_text, tokenizer)
encoded_tensor

tensor([[6109, 3626, 6100,  345, 5699,  284]])

In [13]:
decoded_text =token_ids_to_text(encoded_tensor, tokenizer)
decoded_text

'Every effort moves you closer to'

In [14]:
generated_token_ids = generate_text_simplified(
                                model = model,
                                idx = encoded_tensor,
                                max_new_tokens = 100,
                                context_size = GPT_CONFIG_124M["context_length"]
                            )

In [16]:
#with CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
encoded_tensor = encoded_tensor.to(device)

In [17]:
#with CUDA
generated_token_ids = generate_text_simplified(
                                model = model,
                                idx = encoded_tensor,
                                max_new_tokens = 100,
                                context_size = GPT_CONFIG_124M["context_length"]
                            )

In [19]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107, 588, 11311]])  #  " really like chocolate"]

In [22]:
inputs = inputs.to(device)

with torch.no_grad():
    logits = model(inputs)
token_probas = torch.softmax(logits, dim=-1)


In [24]:
token_probas.shape

torch.Size([2, 3, 50257])

In [26]:
pred_token_ids = torch.argmax(token_probas, dim = -1, keepdim = True)
pred_token_ids

tensor([[[24882],
         [37902],
         [ 9639]],

        [[26385],
         [38031],
         [35452]]], device='cuda:0')

In [34]:
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1:"
      f" {token_ids_to_text(pred_token_ids[0].flatten(), tokenizer)}")

Targets batch 1:  effort moves you
Outputs batch 1: orously UR loaded


In [33]:
pred_token_ids[0].flatten().squeeze(0)

tensor([24882, 37902,  9639], device='cuda:0')

In [35]:
token_probas.shape

torch.Size([2, 3, 50257])

In [41]:
text_idx = 0
target_probas_1 = token_probas[text_idx, [0,1,2], targets[text_idx]]
print("Text 1:", target_probas_1)

text_idx = 1
target_probas_2 = token_probas[text_idx, [0,1,2], targets[text_idx]]
print("Text 2:", target_probas_2)

Text 1: tensor([3.9628e-05, 1.8266e-05, 2.0112e-05], device='cuda:0')
Text 2: tensor([7.5311e-06, 2.2321e-05, 3.2927e-05], device='cuda:0')


In [45]:
target_probas = torch.cat((target_probas_1, target_probas_2))

In [46]:
target_probas

tensor([3.9628e-05, 1.8266e-05, 2.0112e-05, 7.5311e-06, 2.2321e-05, 3.2927e-05],
       device='cuda:0')

In [48]:
log_probas = torch.log(target_probas)
log_probas

tensor([-10.1360, -10.9105, -10.8142, -11.7965, -10.7100, -10.3212],
       device='cuda:0')

In [49]:
avg_log_probas = torch.mean(log_probas, dim = -1)
avg_log_probas

tensor(-10.7814, device='cuda:0')

In [51]:
loss = (-1) * avg_log_probas

In [52]:
print("Logits shape:", token_probas.shape)
print("Targets shape:", targets.shape)

Logits shape: torch.Size([2, 3, 50257])
Targets shape: torch.Size([2, 3])


In [53]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()
print("Flattened logits:", logits_flat.shape)
print("Flattened targets:", targets_flat.shape)

Flattened logits: torch.Size([6, 50257])
Flattened targets: torch.Size([6])


In [56]:
loss_ce = torch.nn.functional.cross_entropy(logits_flat.to(device), targets_flat.to(device))
loss_ce

tensor(10.7814, device='cuda:0')